# Graph Convolutional Neural Network

In [98]:
batch_size = 8 
data_files = [
    {"label": [0], "topology_filename": "../FP11_monomer/SYSTEMns.prmtop", "coordinates_filename": "../FP11_monomer/PRODUCTION_0/TRAJ-150ns.nc"},
    {"label": [0], "topology_filename": "../FP11_monomer/SYSTEMns.prmtop", "coordinates_filename": "../FP11_monomer/PRODUCTION_1/TRAJ-150ns.nc"},
    {"label": [0], "topology_filename": "../FP11_monomer/SYSTEMns.prmtop", "coordinates_filename": "../FP11_monomer/PRODUCTION_2/TRAJ-150ns.nc"},
    {"label": [1], "topology_filename": "../FP12-TLR4-MDago_monomer/SYSTEMns.prmtop", "coordinates_filename": "../FP12-TLR4-MDago_monomer/PRODUCTION_0/TRAJ-150ns.nc"},
    {"label": [1], "topology_filename": "../FP12-TLR4-MDago_monomer/SYSTEMns.prmtop", "coordinates_filename": "../FP12-TLR4-MDago_monomer/PRODUCTION_1/TRAJ-150ns.nc"},
    {"label": [1], "topology_filename": "../FP12-TLR4-MDago_monomer/SYSTEMns.prmtop", "coordinates_filename": "../FP12-TLR4-MDago_monomer/PRODUCTION_2/TRAJ-150ns.nc"}
]
selection_string = "resid 613:754 and name CA N C"
epochs = 15
test_size = 0.20
random_state = 42
lr=0.0001

In [99]:
data_files

[{'label': [0],
  'topology_filename': '../FP11_monomer/SYSTEMns.prmtop',
  'coordinates_filename': '../FP11_monomer/PRODUCTION_0/TRAJ-150ns.nc'},
 {'label': [0],
  'topology_filename': '../FP11_monomer/SYSTEMns.prmtop',
  'coordinates_filename': '../FP11_monomer/PRODUCTION_1/TRAJ-150ns.nc'},
 {'label': [0],
  'topology_filename': '../FP11_monomer/SYSTEMns.prmtop',
  'coordinates_filename': '../FP11_monomer/PRODUCTION_2/TRAJ-150ns.nc'},
 {'label': [1],
  'topology_filename': '../FP12-TLR4-MDago_monomer/SYSTEMns.prmtop',
  'coordinates_filename': '../FP12-TLR4-MDago_monomer/PRODUCTION_0/TRAJ-150ns.nc'},
 {'label': [1],
  'topology_filename': '../FP12-TLR4-MDago_monomer/SYSTEMns.prmtop',
  'coordinates_filename': '../FP12-TLR4-MDago_monomer/PRODUCTION_1/TRAJ-150ns.nc'},
 {'label': [1],
  'topology_filename': '../FP12-TLR4-MDago_monomer/SYSTEMns.prmtop',
  'coordinates_filename': '../FP12-TLR4-MDago_monomer/PRODUCTION_2/TRAJ-150ns.nc'}]

In [100]:
import torch
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

from torch.nn import Embedding, CrossEntropyLoss, Linear, ReLU
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, GATConv
from torch_geometric.nn import radius_graph
from torch_geometric.utils import scatter
from sklearn.model_selection import train_test_split

import MDAnalysis as mda
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import tqdm
import json

## Data Load

In [101]:
# u = mda.Universe("../data/md-2025-03-11/FP11ns.prmtop", "../data/md-2025-03-11/FP11-150ns.nc")
# residue_index = dict((key, id) for id, key in enumerate(np.unique(u.select_atoms('name CA').resnames).tolist()))
# atom_type_index = dict((key, id) for id, key in enumerate(np.unique(u.select_atoms('protein').atoms.types.tolist())))

with open('atom-list.json') as f: 
    atom_type_index = json.load(f)
with open('residue-list.json') as f: 
    residue_index = json.load(f)

In [102]:
def trajectory_to_data(u, selection_string, residue_index, atom_type_index, label):
    for fr in u.trajectory:
        x = torch.from_numpy(u.select_atoms(selection_string).positions)
        residues = list(map(lambda x: residue_index[x], u.select_atoms(selection_string).resnames))
        atom_types = list(map(lambda x: atom_type_index[x], u.select_atoms(selection_string).atoms.types.tolist()))
        d = Data(pos=x, res=torch.tensor(residues, dtype=torch.long), types=torch.tensor(atom_types, dtype=torch.long), label=torch.tensor(label, dtype=torch.long))
        yield d

In [103]:
trj = []

for file in data_files:

    u = mda.Universe(file['topology_filename'], file['coordinates_filename'])
    trj += list(trajectory_to_data(u, selection_string, residue_index, atom_type_index, file['label']))
    print(file['coordinates_filename']) # to check all files are read correctly
    print(len(u.trajectory)) # to check there are 1500 frames per file
    
trj[:5]

../FP11_monomer/PRODUCTION_0/TRAJ-150ns.nc
1500
../FP11_monomer/PRODUCTION_1/TRAJ-150ns.nc
1500
../FP11_monomer/PRODUCTION_2/TRAJ-150ns.nc
1500
../FP12-TLR4-MDago_monomer/PRODUCTION_0/TRAJ-150ns.nc
1500
../FP12-TLR4-MDago_monomer/PRODUCTION_1/TRAJ-150ns.nc
1500
../FP12-TLR4-MDago_monomer/PRODUCTION_2/TRAJ-150ns.nc
1500


[Data(pos=[426, 3], res=[426], types=[426], label=[1]),
 Data(pos=[426, 3], res=[426], types=[426], label=[1]),
 Data(pos=[426, 3], res=[426], types=[426], label=[1]),
 Data(pos=[426, 3], res=[426], types=[426], label=[1]),
 Data(pos=[426, 3], res=[426], types=[426], label=[1])]

## Model

In [104]:
class GCN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.rg = radius_graph
        self.embed_res = Embedding(num_embeddings=23, embedding_dim=128)
        self.embed_type = Embedding(num_embeddings=33, embedding_dim=128)
        #
        self.conv1_res = GCNConv(128, 64)
        self.relu1_res = ReLU()
        self.conv2_res = GCNConv(64, 32)
        self.relu2_res = ReLU()
        self.conv3_res = GATConv(32, 16)
        # 
        self.conv1_type = GCNConv(128, 64)
        self.relu1_type = ReLU()
        self.conv2_type = GCNConv(64, 32)
        self.relu2_type = ReLU()
        self.conv3_type = GCNConv(32, 16)
        # 
        self.linear1 = Linear(32, 2)

    def forward(self, data):
        pos, res, type, batch = data.pos, data.res, data.types, data.batch
        edge_index = radius_graph(pos, r=5.0, batch=batch)
        #
        z_res = self.embed_res(res)
        z_res = self.conv1_res(z_res, edge_index)
        z_res = self.relu1_res(z_res)
        z_res = self.conv2_res(z_res, edge_index)
        z_res = self.relu2_res(z_res)
        z_res = self.conv3_res(z_res, edge_index)
        #
        z_type = self.embed_type(type)
        z_type = self.conv1_type(z_type, edge_index)
        z_type = self.relu1_type(z_type)
        z_type = self.conv2_type(z_type, edge_index)
        z_type = self.relu2_type(z_type)
        z_type = self.conv3_type(z_type, edge_index)
        #
        z_res = scatter(z_res, data.batch, dim=0, reduce='mean')
        z_type = scatter(z_type, data.batch, dim=0, reduce='mean')
        z = torch.cat([z_res, z_type], dim=1)
        z = self.linear1(z)
        z = F.softmax(z, dim=1)
        return z

In [105]:
train_trj, test_trj = train_test_split(trj, test_size=test_size, random_state=random_state)

In [106]:
print(len(train_trj))

7200


In [107]:
print(len(test_trj))

1800


In [108]:
gcn = GCN()
loss = CrossEntropyLoss()
optimizer = torch.optim.Adam(gcn.parameters(), lr=lr, weight_decay=5e-4)
train_loader = DataLoader(train_trj, batch_size=batch_size)
test_loader = DataLoader(test_trj, batch_size=batch_size)
history = []

for i in tqdm.tqdm(range(epochs)):
    statistics = dict()
    statistics['training_loss'] = 0.0
    for d in train_loader:
        z = gcn(d)
        l = loss(z, d.label)
        l.backward()
        optimizer.step()
    statistics['training_loss'] += l.item()
    with torch.no_grad():
        statistics['test_accuracy'] = 0.0
        statistics['test_loss'] = 0.0
        for d in test_loader:
            z = gcn(d)
            l = loss(z, d.label)
            a = ((gcn(d).argmax(1) == d.label).sum())
            statistics['test_accuracy'] += a.item()
            statistics['test_loss'] += l.item()
    statistics['test_accuracy'] /= len(test_trj)
    statistics['test_loss'] /= ((len(test_trj) // batch_size) + 1)
    
    history.append(statistics)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 15/15 [22:26<00:00, 89.75s/it]


In [109]:
pd.DataFrame.from_records(history)

,training_loss,test_accuracy,test_loss
0,0.684818,0.508889,0.688940
1,0.719332,0.491111,0.695182
2,0.660619,0.508889,0.688962
3,0.582747,0.648889,0.638720
4,0.427801,0.690000,0.587410
5,0.352186,0.802222,0.500393
6,0.317672,0.791667,0.512595
7,0.314390,0.821667,0.487560
8,0.313263,0.828333,0.479389
9,0.313262,0.833889,0.474876


In [ ]:
CHANGE: file['label'] en trj[]
    training_loss	test_accuracy	test_loss
0	0.723250	0.476667	0.690948
1	0.718878	0.503333	0.685105
2	0.691495	0.528333	0.684803
3	0.699206	0.485000	0.686742
4	0.704904	0.500000	0.683290

CHANGE: num_embeddings, data.types, embed_res en el modelo GCN
    training_loss	test_accuracy	test_loss
0	0.716966	0.478333	0.686792
1	0.709214	0.478333	0.685698
2	0.690240	0.521667	0.683328
3	0.685908	0.521667	0.681541
4	0.673925	0.521667	0.675898

CHANGE: r=5.0 en radius_graph
	training_loss	test_accuracy	test_loss
0	0.683543	0.521667	0.683303
1	0.722415	0.478333	0.687875
2	0.682698	0.521667	0.683244
3	0.698991	0.478333	0.684323
4	0.690058	0.650000	0.682285

CHANGE: embedding_dim=128, add one layer more en el modelo GCN
    training_loss	test_accuracy	test_loss
0	0.694681	0.478333	0.684095
1	0.713196	0.478333	0.686319
2	0.688857	0.521667	0.683046
3	0.671171	0.521667	0.677181
4	0.619244	0.703333	0.647968 # GREAT IMPROVEMENT

CHANGE: epochs=10
    training_loss	test_accuracy	test_loss
0	0.661112	0.521667	0.683127
1	0.691325	0.521667	0.683531
2	0.678761	0.521667	0.679893
3	0.747541	0.478333	0.677055
4	0.677359	0.550000	0.651588
5	0.572011	0.876667	0.573381
6	0.432996	0.890000	0.456766
7	0.505893	0.783333	0.497697
8	0.404573	0.885000	0.418151
9	0.363043	0.866667	0.457289

CHANGE: lr=0.001, scheduler=ReduceLROnPlateau
    training_loss	test_accuracy	test_loss
0	0.704842	0.478333	0.685103
1	0.700540	0.478333	0.684675
2	0.699476	0.478333	0.684591
3	0.696566	0.478333	0.684328
4	0.699578	0.478333	0.684630
5	0.696926	0.478333	0.684372
6	0.697494	0.478333	0.684429
7	0.697679	0.478333	0.684449
8	0.697763	0.478333	0.684458
9	0.697820	0.478333	0.684464

CHANGE: batch_size=16
    training_loss	test_accuracy	test_loss
0	0.697080	0.478333	0.693441
1	0.706395	0.478333	0.696554
2	0.682156	0.521667	0.691977
3	0.702144	0.478333	0.695240
4	0.688418	0.521667	0.692162
5	0.700152	0.478333	0.694276
6	0.699310	0.478333	0.693346
7	0.683134	0.521667	0.688179
8	0.680758	0.866667	0.682593
9	0.686287	0.478333	0.675832

CHANGE: back to best parameters
    training_loss	test_accuracy	test_loss
0	0.667036	0.521667	0.683151
1	0.683786	0.521667	0.683159
2	0.705477	0.478333	0.683596
3	0.668021	0.568333	0.670443
4	0.653186	0.688333	0.619388
5	0.793520	0.478333	0.660983
6	0.508823	0.945000	0.520236
7	0.391890	0.956667	0.427607
8	0.394068	0.870000	0.442415
9	0.321525	0.948333	0.368391

CHANGE: adding one more file of each simulation
    training_loss	test_accuracy	test_loss
0	0.687449	0.489167	0.688879
1	0.710733	0.510833	0.688338
2	0.668712	0.489167	0.689642
3	0.721171	0.510833	0.681440
4	0.562854	0.489167	0.673841
5	0.778656	0.544167	0.619004
6	0.478034	0.670000	0.586092
7	0.710936	0.792500	0.507526
8	0.510095	0.922500	0.402110
9	0.507958	0.929167	0.382990

CHANGE: adding one more file of each simulation
    training_loss	test_accuracy	test_loss
0	0.697647	0.491111	0.690516
1	0.692313	0.508889	0.690027
2	0.688565	0.508889	0.689923
3	0.675422	0.508889	0.692195
4	0.690053	0.508889	0.689934
5	0.712709	0.491111	0.693659
6	0.690835	0.508889	0.689962
7	0.672896	0.508889	0.693423
8	0.712729	0.491111	0.693631
9	0.697781	0.491111	0.690572

CHANGE: batch_size=16 lots of samples
	training_loss	test_accuracy	test_loss
0	0.695721	0.491111	0.693320
1	0.698120	0.491111	0.693698
2	0.691714	0.508889	0.693052
3	0.695765	0.491111	0.693401
4	0.686243	0.508889	0.693029
5	0.705793	0.491111	0.695099
6	0.679319	0.508889	0.693954
7	0.702882	0.491111	0.694462
8	0.690761	0.508889	0.693017
9	0.686886	0.508889	0.693004

CHANGE: batch_size=16, scheduler=ReduceLROnPlateau
    training_loss	test_accuracy	test_loss
0	0.683435	0.508889	0.693252
1	0.684391	0.508889	0.693089
2	0.685384	0.508889	0.692969
3	0.685994	0.508889	0.692911
4	0.686938	0.508889	0.692860
5	0.692000	0.508889	0.692911
6	0.692070	0.508889	0.692915
7	0.692143	0.508889	0.692922
8	0.692236	0.508889	0.692931
9	0.693206	0.741111	0.693005

CHANGE: batch_size=8, scheduler=ReduceLROnPlateau
	training_loss	test_accuracy	test_loss
0	0.683859	0.508889	0.690132
1	0.685782	0.508889	0.689946
2	0.687104	0.508889	0.689879
3	0.688200	0.508889	0.689860
4	0.689515	0.508889	0.689869
5	0.692176	0.508889	0.689963
6	0.692370	0.508889	0.689980
7	0.692481	0.508889	0.689993
8	0.692557	0.508889	0.690004
9	0.693112	0.527778	0.690045

CHANGE: add one more layer
    training_loss	test_accuracy	test_loss
0	0.701876	0.491111	0.691176
1	0.700987	0.491111	0.691037
2	0.680704	0.508889	0.690716
3	0.683955	0.508889	0.690168
4	0.682823	0.508889	0.690297
5	0.678118	0.508889	0.691275
6	0.685027	0.508889	0.690075
7	0.692176	0.508889	0.690016
8	0.703745	0.491111	0.691571
9	0.692062	0.508889	0.690022

CHANGE: selection_string="name CA N C"
	training_loss	test_accuracy	test_loss
0	0.646434	0.521667	0.684665
1	0.674309	0.521667	0.683081
2	0.669784	0.521667	0.683020
3	0.697761	0.478333	0.684198
4	0.711502	0.478333	0.684663
5	0.685733	0.643333	0.670426
6	0.585218	0.548333	0.628553
7	0.650042	0.703333	0.588664
8	0.494995	0.636667	0.571829
9	0.453853	0.886667	0.472910

CHANGE: best parameters epochs=15 to see if the model stabilizes
    training_loss	test_accuracy	test_loss
0	0.694230	0.478333	0.684041
1	0.704741	0.478333	0.685018
2	0.710157	0.478333	0.684907
3	0.671046	0.893333	0.670169
4	0.639921	0.686667	0.619434
5	0.495741	0.581667	0.590689
6	0.500573	0.968333	0.512210
7	0.356240	0.950000	0.379944
8	0.316537	0.918333	0.380767
9	0.458167	0.866667	0.435892
10	0.523144	0.820000	0.493047
11	0.449344	0.840000	0.458231
12	0.411325	0.893333	0.424755
13	0.352580	0.945000	0.379284
14	0.313807	0.958333	0.349356

CHANGE: full set of simulations
    training_loss	test_accuracy	test_loss
0	0.668653	0.508889	0.696778
1	0.688952	0.508889	0.689923
2	0.712364	0.491111	0.693562
3	0.681031	0.508889	0.690659
4	0.695410	0.491111	0.690089
5	0.693216	0.492222	0.690131
6	0.672193	0.508889	0.693789
7	0.678082	0.508889	0.691307
8	0.724449	0.491111	0.697259
9	0.688262	0.508889	0.800814
10	0.688262	0.508889	0.800814
11	0.688262	0.508889	0.800814
12	0.688262	0.508889	0.800814
13	0.688262	0.508889	0.800814
14	0.688262	0.508889	0.800814

CHANGE: selection_sring="name CA N C"
    training_loss	test_accuracy	test_loss
0	0.689883	0.521667	0.683623
1	0.705725	0.478333	0.685172
2	0.711298	0.478333	0.685480
3	0.691008	0.511667	0.678663
4	0.603770	0.521667	0.661864
5	0.508965	0.725000	0.565724
6	0.456539	0.758333	0.532928
7	0.446959	0.680000	0.531519
8	0.383040	0.948333	0.414420
9	0.374402	0.881667	0.434833
10	0.333302	0.990000	0.335889
11	0.316180	0.941667	0.365968
12	0.372379	0.948333	0.365250
13	0.313353	0.955000	0.354632
14	0.352472	0.973333	0.333593

CHANGE: full simulation
	training_loss	test_accuracy	test_loss
0	0.681583	0.508889	0.690520
1	0.705233	0.491111	0.691837
2	0.694638	0.491111	0.690194
3	0.700117	0.491111	0.690298
4	0.721952	0.491111	0.696624
5	0.705501	0.535000	0.682825
6	0.736349	0.627222	0.661002
7	0.637689	0.624444	0.654211
8	0.694686	0.656111	0.641117
9	0.717229	0.656111	0.646868
10	0.688420	0.660000	0.641415
11	0.688607	0.667222	0.638776
12	0.697827	0.670000	0.633896
13	0.690048	0.672778	0.632244
14	0.702072	0.676111	0.626015

CHANGE: selection_string="resid 613:754 and name CA N C"
    training_loss	test_accuracy	test_loss
0	0.652031	0.521667	0.672299
1	0.473067	0.883333	0.495463
2	0.932859	0.478333	0.752425
3	0.373286	0.940000	0.416505
4	0.517523	0.750000	0.528364
5	0.390173	0.916667	0.389213
6	0.319851	0.946667	0.362166
7	0.402974	0.823333	0.482555
8	0.315449	0.966667	0.342981
9	0.313992	0.968333	0.340302
10	0.313262	0.961667	0.347585
11	0.320961	0.960000	0.351645
12	0.313262	0.963333	0.343261
13	0.313262	0.966667	0.341309
14	0.313262	0.966667	0.340405

CHANGE: full simulations, selection_string="resid 613:754 and name CA N C" 
	training_loss	test_accuracy	test_loss
0	0.684818	0.508889	0.688940
1	0.719332	0.491111	0.695182
2	0.660619	0.508889	0.688962
3	0.582747	0.648889	0.638720
4	0.427801	0.690000	0.587410
5	0.352186	0.802222	0.500393
6	0.317672	0.791667	0.512595
7	0.314390	0.821667	0.487560
8	0.313263	0.828333	0.479389
9	0.313262	0.833889	0.474876
10	0.313262	0.796667	0.508356
11	0.313290	0.821667	0.487238
12	0.313262	0.853889	0.455966
13	0.313262	0.867222	0.443051
14	0.313262	0.895000	0.414541
